# Dataset Characteristics Analysis

This notebook analyzes the characteristics of the evaluation dataset, which consists of projects from three sources:

1. **EqBench**: Programs from the EqBench benchmark designed for automated reasoning tools, with EvoSuite-generated test suites at different timeout settings (1s, 10s, 60s)
2. **Apache Commons Utils**: Utility methods extracted from Apache Commons projects, including both developer-written tests and EvoSuite-generated test suites
3. **RepoReapers**: Open source Java projects from the RepoReapers dataset, selected based on size, structure, and build tool criteria

The analysis provides file counts, class counts, source lines of code (SLOC), and test method statistics for each project.

In [ ]:
# Import dataset analysis functions
from teralizer.dataset_characteristics import (
    get_dataset_statistics,
    generate_dataset_table,
    generate_dataset_csv_data,
)
from teralizer.config import db_config
from teralizer.exclusions import get_excluded_project_names
from teralizer.exports import save_latex_table, save_csv_data
from IPython.display import display

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## Data Collection

Collect statistics for all projects in the evaluation dataset.

In [ ]:
# Get database connections
conn_dev = db_config.get_dev_engine()
conn_test = db_config.get_test_engine()

# Get excluded project names from both databases
excluded_projects_dev = get_excluded_project_names(conn_dev)
excluded_projects_test = get_excluded_project_names(conn_test)
excluded_projects = excluded_projects_dev.union(excluded_projects_test)
print(f"Excluding {len(excluded_projects)} projects from dataset statistics")

# Get dataset statistics (computes fresh if projects/ available, else loads pre-computed)
project_aggregates = get_dataset_statistics(conn_dev, conn_test, excluded_projects)

# Display aggregated statistics
display(project_aggregates)

## LaTeX Table Generation

Generate the dataset characteristics table for paper inclusion.

In [ ]:
# Generate LaTeX table
latex_table = generate_dataset_table(project_aggregates)
print(latex_table)

# Save LaTeX table
save_latex_table(latex_table, "tab-dataset-statistics")

## CSV Data Export

Export dataset statistics as CSV for further analysis.

In [ ]:
# Generate CSV data
csv_data = generate_dataset_csv_data(project_aggregates)

# Save CSV data
csv_path = save_csv_data(
    csv_data,
    "dataset-statistics-data",
    "Dataset statistics showing files, classes, SLOC, and test methods per project",
)

print(f"Dataset statistics exported to: {csv_path}")
print(f"Shape: {csv_data.shape}")
print("Sample data:")
display(csv_data.head())

## Dataset Summary

The evaluation dataset includes:

- **EqBench projects**: Programs designed for automated reasoning tools, avoiding features like recursion and reflection that challenge symbolic execution
- **Apache Commons projects**: Utility methods extracted from Apache Commons libraries, representing larger open source codebases
- **RepoReapers projects**: Selection of Java projects meeting specific size and structure criteria for broader applicability assessment

Each project type uses different test suite generation approaches (EvoSuite with varying timeouts, developer-written tests) to evaluate generalization effectiveness across different testing scenarios.